# Tarrant CAD — PropertyData_R_2025 ETL

Exploration / processing notebook for the Tarrant Appraisal District residential export `PropertyData_R_2025.txt`.

**Source format** (checked on the 2025 file):
- Pipe-delimited (`|`), CRLF line endings, 56 columns, ~659K rows
- Values are space-padded fixed-width inside the pipes (e.g. `|      110000|`)
- One row per account (`Record_Type = AAAA`)
- Sentinel values: deed date `12-31-1900`, deed book/page `0000000`, instrument `0000000000`, year built `0`

Steps: **Load raw → Clean → Type → Profile → Validate → Export (bronze)**.

## 1. Setup

In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path("../Data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "PropertyData_2025.txt"
print(INPUT_FILE.resolve(), f"{INPUT_FILE.stat().st_size / 1e6:,.1f} MB")

/Users/user2/PTP/ptp-data/tarrant/Data/PropertyData_2025.txt 779.2 MB


## 2. Load raw
Read everything as string first so nothing (leading zeros in `Account_Num`, zips, codes) gets mangled by type inference.

In [2]:
def show_bad(bad_line):
    print("BAD LINE:", "|".join(bad_line))
    return None  # skip the row

raw = pd.read_csv(
    INPUT_FILE,
    sep="|",
    dtype=str,
    keep_default_na=False,   # keep blanks as "" until we clean them ourselves
    encoding="latin-1",      # safe for any stray non-UTF-8 bytes
    quoting=3,               # csv.QUOTE_NONE: owner names / legal descriptions can contain quotes
    engine="python",       # required for a callable on_bad_lines
    on_bad_lines=show_bad,

)
raw.columns = raw.columns.str.strip()
print(raw.shape)
raw.head()

(1996478, 56)


,RP,Appraisal_Year,Account_Num,Record_Type,Sequence_No,PIDN,Owner_Name,Owner_Address,Owner_CityState,Owner_Zip,Owner_Zip4,Owner_CRRT,Situs_Address,Property_Class,TAD_Map,MAPSCO,Exemption_Code,State_Use_Code,LegalDescription,Notice_Date,County,City,School,Num_Special_Dist,Spec1,Spec2,Spec3,Spec4,Spec5,Deed_Date,Deed_Book,Deed_Page,Land_Value,Improvement_Value,Total_Value,Garage_Capacity,Num_Bedrooms,Num_Bathrooms,Year_Built,Living_Area,Swimming_Pool_Ind,ARB_Indicator,Ag_Code,Land_Acres,Land_SqFt,Ag_Acres,Ag_Value,Central_Heat_Ind,Central_Air_Ind,Structure_Count,From_Accts,Appraisal_Date,Appraised_Value,GIS_Link,Instrument_No,Overlap_Flag
0,C,2025,00000051,AAAA,000,14437----2----1,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,117 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 1,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,450000,0,450000,0,0,0,0,0,,,,0.1147,5000,0.0000,000000000,N,N,0,,01-01-2025,450000,14437-2-1,0000000000,
1,C,2025,00000078,AAAA,000,14437----2----2,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,111 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 2,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,450000,0,450000,0,0,0,0,0,,,,0.1147,5000,0.0000,000000000,N,N,0,,01-01-2025,450000,14437-2-2,0000000000,
2,C,2025,00000086,AAAA,000,14437----2----3A--20,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,101 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 3A & 4,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,675000,0,675000,0,0,0,0,0,,,,0.1721,7500,0.0000,000000000,N,N,0,,01-01-2025,675000,14437-2-3A,0000000000,
3,C,2025,00000094,AAAA,000,14437----2----3B,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,115 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 3B,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,225000,0,225000,0,0,0,0,0,,,,0.0573,2500,0.0000,000000000,N,N,0,,01-01-2025,225000,14437-2-3B,0000000000,
4,C,2025,00000108,AAAA,000,14437----3----1---20,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,100 W BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 3 Lot 1 & 2,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,1080000,0,1080000,0,0,0,0,0,,,,0.2754,12000,0.0000,000000000,N,N,0,,01-01-2025,1080000,14437-3-1,0000000000,


## 3. Clean strings
Strip padding and turn empty strings into nulls.

In [3]:
raw.describe(include="all").T

,count,unique,top,freq
RP,1996478,4,M,1207764
Appraisal_Year,1996478,1,2025,1996478
Account_Num,1996478,1996478,00000051,1
Record_Type,1996478,1,AAAA,1996478
Sequence_No,1996478,1,000,1996478
PIDN,1996478,1952334,04243900,335
Owner_Name,1996478,768630,BKV NORTH TEXAS LLC,4775
Owner_Address,1996478,632426,,12489
Owner_CityState,1996478,7531,"FORT WORTH, TX",645617
Owner_Zip,1996478,11201,76063,55256


In [4]:
raw.to_csv(OUTPUT_DIR / "PropertyData_2025.csv", index=False)